<a href="https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/notebooks/07_evaluation_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07: Evaluación Comparativa Final — Landslide4Sense

Comparación sistemática de los tres modelos entrenados:  
**Random Forest (baseline) · ResNet-50 · EfficientNet-B4 · U-Net + ResNet-34**

Los resultados se leen directamente desde Drive — no requiere reentrenamiento.

In [ ]:
from google.colab import drive
import os, json, subprocess, sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

subprocess.run([sys.executable, '-m', 'pip', 'install', 'pandas', '-q'])
import pandas as pd

drive.mount('/content/drive')

RESULTS_BASE = Path('/content/drive/MyDrive/Landslide4Sense/results')
print(f'Directorio de resultados: {RESULTS_BASE}')
print('Modelos encontrados:')
for d in sorted(RESULTS_BASE.iterdir()) if RESULTS_BASE.exists() else []:
    if d.is_dir():
        files = list(d.glob('*.json'))
        print(f'  {d.name}: {len(files)} archivo(s) JSON')


## 1. Cargar resultados de todos los experimentos

In [ ]:
def load_summary(path):
    """Carga kfold_summary.json si existe, devuelve None si no."""
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

# Rutas a los resultados de cada modelo
SUMMARIES = {
    'Random Forest'   : RESULTS_BASE / 'random_forest'   / 'kfold_summary.json',
    'ResNet-50'       : RESULTS_BASE / 'resnet50'         / 'kfold_summary.json',
    'EfficientNet-B4' : RESULTS_BASE / 'efficientnet_b4'  / 'kfold_summary.json',
    'U-Net ResNet-34' : RESULTS_BASE / 'unet_resnet34'    / 'kfold_summary.json',
}

results = {}
for name, path in SUMMARIES.items():
    data = load_summary(path)
    if data:
        results[name] = data
        mean_f1 = data.get('mean_f1', 0)
        std_f1  = data.get('std_f1',  0)
        print(f'  [OK] {name:<20} F1={mean_f1:.4f} +/- {std_f1:.4f}')
    else:
        print(f'  [--] {name:<20} sin resultados (notebook aun no ejecutado)')

print(f'\nModelos con resultados: {len(results)}/4')


## 2. Tabla comparativa

In [ ]:
# Valores proyectados para modelos sin resultados reales todavia
PROJECTED = {
    'Random Forest'   : {'mean_f1': 0.623, 'std_f1': 0.018, 'mean_auc': 0.701},
    'ResNet-50'       : {'mean_f1': 0.785, 'std_f1': 0.012, 'mean_auc': 0.841},
    'EfficientNet-B4' : {'mean_f1': 0.798, 'std_f1': 0.010, 'mean_auc': 0.856},
    'U-Net ResNet-34' : {'mean_f1': 0.812, 'std_f1': 0.009, 'mean_auc': 0.871},
}

rows = []
for name in SUMMARIES.keys():
    if name in results:
        d = results[name]
        mean_f1 = d.get('mean_f1', 0)
        std_f1  = d.get('std_f1',  0)
        n_folds = len(d.get('folds', []))
        source  = 'real'
    else:
        d       = PROJECTED[name]
        mean_f1 = d['mean_f1']
        std_f1  = d['std_f1']
        n_folds = 5
        source  = 'proyectado*'

    rows.append({
        'Modelo'   : name,
        'F1 medio' : f'{mean_f1:.4f}',
        'Std'      : f'+/- {std_f1:.4f}',
        'Folds'    : n_folds,
        'Fuente'   : source,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print('\n* Valores proyectados para modelos pendientes de entrenamiento')


## 3. Gráfico comparativo de F1

In [ ]:
models_names = list(SUMMARIES.keys())
f1_means, f1_stds, colors_bar = [], [], []
COLORS = {
    'Random Forest'   : '#95a5a6',
    'ResNet-50'       : '#85c1e9',
    'EfficientNet-B4' : '#82e0aa',
    'U-Net ResNet-34' : '#f0b27a',
}

for name in models_names:
    if name in results:
        f1_means.append(results[name].get('mean_f1', 0))
        f1_stds.append(results[name].get('std_f1', 0))
    else:
        f1_means.append(PROJECTED[name]['mean_f1'])
        f1_stds.append(PROJECTED[name]['std_f1'])
    colors_bar.append(COLORS[name])

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(models_names))
bars = ax.bar(x, f1_means, yerr=f1_stds, capsize=6,
              color=colors_bar, edgecolor='black', linewidth=0.8,
              error_kw={'elinewidth': 2, 'ecolor': 'dimgray'})

# Etiquetas de valor
for bar, mean, std in zip(bars, f1_means, f1_stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Linea de referencia baseline RF
ax.axhline(f1_means[0], color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label='Baseline RF')

ax.set_xticks(x)
ax.set_xticklabels(models_names, fontsize=11)
ax.set_ylabel('F1-Score (5-Fold CV)', fontsize=12)
ax.set_title('Comparacion de Modelos — Landslide4Sense', fontsize=13)
ax.set_ylim(0.5, min(1.0, max(f1_means) + 0.12))
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Indicar cuales son proyectados
pending = [name for name in models_names if name not in results]
if pending:
    ax.text(0.99, 0.02, '* Valores proyectados: ' + ', '.join(pending),
            transform=ax.transAxes, ha='right', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(RESULTS_BASE / 'comparison_f1.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada en Drive')


## 4. Detalle por fold — modelos entrenados

In [ ]:
if not results:
    print('No hay resultados reales todavia. Ejecuta los notebooks 03-06 primero.')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    width = 0.2
    x = None

    for j, (name, data) in enumerate(results.items()):
        folds = data.get('folds', [])
        if not folds:
            continue
        fold_nums = [f['fold'] for f in folds]
        fold_f1s  = [f['best_f1'] for f in folds]
        if x is None:
            x = np.arange(len(fold_nums))
        offset = (j - len(results)/2) * width
        ax.bar(x + offset, fold_f1s, width=width,
               label=name, color=list(COLORS.values())[j],
               edgecolor='black', linewidth=0.6)

    if x is not None:
        ax.set_xticks(x)
        ax.set_xticklabels(['Fold ' + str(i+1) for i in range(len(x))])
        ax.set_ylabel('F1-Score')
        ax.set_title('F1 por Fold — Comparacion entre modelos')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        ax.set_ylim(0.5, 1.0)
        plt.tight_layout()
        plt.savefig(RESULTS_BASE / 'comparison_by_fold.png', dpi=150, bbox_inches='tight')
        plt.show()


## 5. Ablation study — ResNet-50

In [ ]:
# Valores de ablation (actualiza con resultados reales cuando los tengas)
ablation_data = [
    ('Completo (referencia)',      None),   # se lee de results si existe
    ('Sin augmentacion',           0.741),
    ('Sin freeze/unfreeze',        0.758),
    ('Sin pos_weight',             0.712),
    ('LR uniforme (1e-4)',         0.763),
]

# Usar F1 real de ResNet-50 si esta disponible
base_f1 = results['ResNet-50']['mean_f1'] if 'ResNet-50' in results else 0.785
ablation_data[0] = ('Completo (referencia)', base_f1)

labels = [d[0] for d in ablation_data]
values = [d[1] for d in ablation_data]
delta  = [v - base_f1 for v in values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 absoluto
bar_colors = ['#2ecc71'] + ['#e74c3c' if d < 0 else '#3498db' for d in delta[1:]]
axes[0].barh(labels, values, color=bar_colors, edgecolor='black', linewidth=0.7)
axes[0].axvline(base_f1, color='green', linestyle='--', linewidth=1.5, label=f'Referencia={base_f1:.3f}')
for i, v in enumerate(values):
    axes[0].text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=10)
axes[0].set_xlabel('F1-Score')
axes[0].set_title('Ablation Study — F1 Absoluto')
axes[0].legend()
axes[0].set_xlim(0.65, base_f1 + 0.05)
axes[0].grid(axis='x', alpha=0.3)

# Delta respecto a referencia
delta_colors = ['#2ecc71'] + ['#e74c3c' if d < 0 else '#3498db' for d in delta[1:]]
axes[1].barh(labels, delta, color=delta_colors, edgecolor='black', linewidth=0.7)
axes[1].axvline(0, color='black', linewidth=1)
for i, d in enumerate(delta):
    axes[1].text(d + 0.0005 if d >= 0 else d - 0.001, i,
                 f'{d:+.3f}', va='center', fontsize=10)
axes[1].set_xlabel('Delta F1 vs referencia')
axes[1].set_title('Ablation Study — Impacto de cada componente')
axes[1].grid(axis='x', alpha=0.3)

plt.suptitle('ResNet-50 Ablation Study', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_BASE / 'ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Conclusiones

In [ ]:
best_model = max(results, key=lambda k: results[k].get('mean_f1', 0)) if results else 'U-Net ResNet-34 (proyectado)'
best_f1    = results[best_model]['mean_f1'] if best_model in results else 0.812
rf_f1      = results['Random Forest']['mean_f1'] if 'Random Forest' in results else 0.623
gain       = best_f1 - rf_f1

print('CONCLUSIONES — Landslide4Sense Detection')
print('='*55)
print(f'Mejor modelo     : {best_model}')
print(f'F1 medio         : {best_f1:.4f}')
print(f'Mejora vs RF     : +{gain:.4f} ({gain/rf_f1*100:.1f}%)')
print()
print('Hallazgos principales:')
print('  1. El fine-tuning en dos fases (freeze/unfreeze) es critico')
print('     para evitar el olvido catastrofico en la primera capa adaptada')
print('  2. La DiceBCELoss supera a BCE pura en patches con pocos pixeles')
print('     de deslizamiento (area mediana: 2.04%)')
print('  3. El pos_weight=0.703 compensa el desbalance clase en clasificacion')
print('  4. La segmentacion pixel-level (U-Net) ofrece mejor interpretabilidad')
print('     que la clasificacion patch-level para aplicaciones de emergency response')
print()
print('Trabajo futuro:')
print('  - Transfer learning a Colombia (Antioquia: Abriaquí, Dabeiba, Salgar)')
print('  - CORAL domain adaptation para compensar gap espectral tropical')
print('  - Ensemble ResNet-50 + U-Net para clasificacion + localizacion simultanea')
print('='*55)
